### 05_enrollment
Enroll Module that enrolls user into system
* Create templates based on merged embeddings for each identity
* Creates FAISS index for users
* Usage: 
    * Build HQ user templates and "enrolls" (i.e. saves to templates folder)
        * `enroll_new_user_hq()` --> `templates_enroll_hq`
        * Build template for 1 person (through webcam data) and append to existing FAISS
            - `build_user_template_hq()`
    * Builds gallery of HQ templates for users in `<dataset>/enroll`. Two ways:
        * Pose-aware: 
            - `build_template_gallery_pose_hq()` --> `templates_pose_hq`
        * Single template per user: 
            - `build_template_gallery_hq()` --> `templates_all_enroll_hq`
* Output:
    * Each returns a `.csv`, `.npy`, and `.index` to `<dataset>/enrolled_users`

In [1]:
# ----- Imports and config -----
from pathlib import Path
import numpy as np
import pandas as pd
import faiss, sys

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.build_templates import (
    build_template_gallery_hq,          # for building HQ templates
    load_gallery_hq,                    # for loading existing template gallery
    enroll_new_user_hq,                 # for enrolling a new user
    build_user_template_hq,             # for single user enrollment
    build_template_gallery_pose_hq,     # for pose-aware HQ templates
    ID_COL,
)

In [2]:
# ----- Helper functions -----

def build_faiss(templates, templates_map, out_dir):
    # Cell 3: L2-normalize templates, build FAISS index, save to enrolled_users

    # L2-normalize
    norms = np.linalg.norm(templates, axis=1, keepdims=True).clip(min=1e-12)
    templates_norm = templates / norms

    # Build FAISS index
    dim = templates_norm.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(templates_norm)

    # Save
    OUT_PREFIX = "templates_enroll_hq"
    tpl_path = out_dir / f"{OUT_PREFIX}.npy"
    map_path = out_dir / f"{OUT_PREFIX}_map.csv"
    idx_path = out_dir / f"{OUT_PREFIX}.index"

    np.save(tpl_path, templates_norm)
    templates_map.to_csv(map_path, index=False)
    faiss.write_index(index, str(idx_path))

    print(f"Saved HQ templates: {tpl_path} (shape={templates_norm.shape})")
    print(f"Saved HQ map:       {map_path} (rows={len(templates_map)})")
    print(f"Saved HQ index:     {idx_path}")


def enroll_users(df, EMB_DIR, embeddings):

    # Build per-person templates from enroll embeddings (mean + L2 normalize), save template_index in map
    # - Builds 1 template per person_id 
    # - Averages all embeddings for that person_id to create the template
    # - Saves templates and templates_map to ../data_processed/vggface2
    # - Builds FAISS index (IndexFlatIP) as templates_enroll.index

    # loop over each user and enroll them with HQ templates
    out_dir = EMB_DIR / "enrolled_users" 
    out_dir.mkdir(parents=True, exist_ok=True)

    templates = []
    rows = []

    for person_id, df_person in df.groupby(ID_COL):
        # This is "enrolling" ONE user using your HQ rules
        tpl = build_user_template_hq(df_person, embeddings)  # (D,)
        templates.append(tpl)
        rows.append(
            {
                ID_COL: person_id,
                "template_index": len(rows),
            }
        )

    templates = np.vstack(templates).astype(np.float32)
    templates_map = pd.DataFrame(rows)
    templates_map["template_index"] = np.arange(len(templates_map))

    print("Enrolled users:", templates_map[ID_COL].nunique())
    print("Templates shape:", templates.shape)
    print(templates_map.head())

    # Build and save FAISS index
    build_faiss(templates, templates_map, out_dir)

    return templates


In [3]:
DATA_ROOT = "../data_processed/vggface2"
SPLIT = "enroll"
EMB_DIR = Path(DATA_ROOT) / "embeddings" / SPLIT
USER_DIR = Path("../data_processed/vggface2")

MAP_CSV = EMB_DIR / "embeddings_map_with_pose_k5.csv"
EMB_NPY = EMB_DIR / "embeddings.npy"

df = pd.read_csv(MAP_CSV)
embeddings = np.load(EMB_NPY).astype(np.float32)

print("Rows in CSV:", len(df))
print("Embeddings shape:", embeddings.shape)

templates = enroll_users(df, USER_DIR, embeddings)

Rows in CSV: 140922
Embeddings shape: (140922, 512)
Enrolled users: 480
Templates shape: (480, 512)
  person_id  template_index
0   n000002               0
1   n000003               1
2   n000004               2
3   n000005               3
4   n000006               4
Saved HQ templates: ..\data_processed\vggface2\enrolled_users\templates_enroll_hq.npy (shape=(480, 512))
Saved HQ map:       ..\data_processed\vggface2\enrolled_users\templates_enroll_hq_map.csv (rows=480)
Saved HQ index:     ..\data_processed\vggface2\enrolled_users\templates_enroll_hq.index


In [4]:
OUT_PREFIX = "templates_all_enroll_hq"

templates, templates_map, index = build_template_gallery_hq(
    split=SPLIT,
    data_root=DATA_ROOT,
    out_name_prefix=OUT_PREFIX,
)

print("HQ gallery built!")
print("Templates shape:", templates.shape)
print("Number of enrolled users:", templates_map[ID_COL].nunique())
print(templates_map.head())

Saved HQ templates: ..\data_processed\vggface2\enrolled_users\templates_all_enroll_hq.npy (shape=(480, 512))
Saved HQ map:       ..\data_processed\vggface2\enrolled_users\templates_all_enroll_hq_map.csv (rows=480)
Saved HQ index:     ..\data_processed\vggface2\enrolled_users\templates_all_enroll_hq.index
HQ gallery built!
Templates shape: (480, 512)
Number of enrolled users: 480
  person_id  template_index
0   n000002               0
1   n000003               1
2   n000004               2
3   n000005               3
4   n000006               4


In [5]:
# Load the gallery to verify
templates, templates_map, index, out_dir = load_gallery_hq(
    split=SPLIT,
    data_root=DATA_ROOT,
    out_name_prefix=OUT_PREFIX,
)

print("Loaded gallery from:", out_dir)
print("Templates shape:", templates.shape)
print("Enrolled IDs:", templates_map[ID_COL].nunique())


Loaded gallery from: ..\data_processed\vggface2\enrolled_users
Templates shape: (480, 512)
Enrolled IDs: 480


In [6]:
# create pose-aware templates
# - Builds multiple templates per person_id, one per pose bin
#     - in this case, 5 bins: front, left, right, up, down
# - Requires embeddings_map with a pose column (falls back to yaw->bin)
# - Saves templates and templates_map to ../data_processed/vggface2/enrolled_users
# - Builds FAISS index (IndexFlatIP) as templates_pose_hq.index

templates_pose, templates_map_pose, index_pose = build_template_gallery_pose_hq(
    split=SPLIT,
    data_root=DATA_ROOT
)
print("Pose-aware HQ gallery built!")

Saved pose HQ templates: ..\data_processed\vggface2\enrolled_users\templates_pose_hq.npy (shape=(2400, 512))
Saved pose HQ map:       ..\data_processed\vggface2\enrolled_users\templates_pose_hq_map.csv (rows=2400)
Saved pose HQ index:     ..\data_processed\vggface2\enrolled_users\templates_pose_hq.index
Pose-aware HQ gallery built!


In [12]:
# Load the gallery to verify all saved templates

for prefix in ["templates_pose_hq", "templates_all_enroll_hq", "templates_enroll_hq"]:

    templates, templates_map, index, out_dir = load_gallery_hq(
        split=SPLIT,
        data_root=DATA_ROOT,
        out_name_prefix=prefix,
    )

    print(f"Loaded gallery '{prefix}' from: {out_dir}")
    print("Templates shape:", templates.shape)
    print("Enrolled IDs:", templates_map[ID_COL].nunique())
    print("")


Loaded gallery 'templates_pose_hq' from: ..\data_processed\vggface2\enrolled_users
Templates shape: (2400, 512)
Enrolled IDs: 480

Loaded gallery 'templates_all_enroll_hq' from: ..\data_processed\vggface2\enrolled_users
Templates shape: (480, 512)
Enrolled IDs: 480

Loaded gallery 'templates_enroll_hq' from: ..\data_processed\vggface2\enrolled_users
Templates shape: (480, 512)
Enrolled IDs: 480

